In [1]:
# Step 0: Install dependencies
!pip install ultralytics opencv-python-headless transformers torch torchvision pillow

# Step 1: Imports
from ultralytics import YOLO
import cv2
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import torch

# Step 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 3: Load pre-trained YOLOv8 model
model = YOLO('yolov8n.pt')  # small model for Colab

# Step 4: Train YOLOv8 on your dataset
model.train(
    data='/content/drive/MyDrive/smart_retail_project/my_dataset.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=0
)

# Step 5: Save trained model
model.save('/content/drive/MyDrive/smart_retail_project/smart_retail_yolov8.pt')

# Step 6: Object detection on an image
image_path = '/content/drive/MyDrive/smart_retail_project/demo/sample_videos/frame1.jpg'
results = model(image_path)
annotated_img = results[0].plot()
Image.fromarray(annotated_img)

# Step 7: Optional - Object detection on video
video_path = '/content/drive/MyDrive/smart_retail_project/demo/sample_videos/store.mp4'
cap = cv2.VideoCapture(video_path)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    results = model(frame)
    annotated_frame = results[0].plot()
    cv2.imshow("Detection", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

# Step 8: Multimodal scene captions using CLIP
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image = Image.open(image_path)
scene_texts = ["person picking item", "empty shelf", "full shelf"]
inputs = processor(text=scene_texts, images=image, return_tensors="pt", padding=True)
outputs = clip_model(**inputs)
logits_per_image = outputs.logits_per_image
predicted_index = logits_per_image.argmax().item()
print(f"Scene description: {scene_texts[predicted_index]}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ultralytics 8.3.217 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/smart_retail_project/my_dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dro

RuntimeError: Dataset '/content/drive/MyDrive/smart_retail_project/my_dataset.yaml' error ❌ '/content/drive/MyDrive/smart_retail_project/my_dataset.yaml' does not exist